In [ ]:

input_data = """5
cat N
watch N
watch V
study V
big ADJ
8
cats
watches
watching
studied
bigger
cat
biggest
elephants"""

data = input_data.splitlines()
i = 0

S = int(data[i])
i += 1

lexicon = []
for _ in range(S):
    stem, pos = data[i].split()
    lexicon.append((stem, pos))
    i += 1

Q = int(data[i])
i += 1

vowels = set("aeiou")

def doublable(stem):
    if len(stem) < 3 or stem[-1] in "wxy":
        return False
    if not (stem[-3] not in vowels and stem[-2] in vowels and stem[-1] not in vowels):
        return False
    groups = sum(
        1 for j in range(len(stem))
        if stem[j] in vowels and (j == 0 or stem[j - 1] not in vowels)
    )
    return groups == 1

def generate(stem, feature):
    if feature in ("PL", "3SG"):
        if stem.endswith(("s", "x", "z", "ch", "sh")):
            return stem + "es"
        if len(stem) >= 2 and stem.endswith("o") and stem[-2] not in vowels:
            return stem + "es"
        if len(stem) >= 2 and stem.endswith("y") and stem[-2] not in vowels:
            return stem[:-1] + "ies"
        return stem + "s"

    if feature == "PROG":
        if len(stem) >= 3 and stem.endswith("e") and stem[-2] not in vowels:
            return stem[:-1] + "ing"
        if doublable(stem):
            return stem + stem[-1] + "ing"
        return stem + "ing"

    suffix = {"PAST": "ed", "COMP": "er", "SUP": "est"}[feature]

    if stem.endswith("e"):
        return stem + suffix[1:]
    if len(stem) >= 2 and stem.endswith("y") and stem[-2] not in vowels:
        return stem[:-1] + "i" + suffix
    if doublable(stem):
        return stem + stem[-1] + suffix
    return stem + suffix

features = {
    "N": ["PL"],
    "V": ["3SG", "PROG", "PAST"],
    "ADJ": ["COMP", "SUP"]
}

for _ in range(Q):
    surface = data[i]
    i += 1
    analyses = set()

    for stem, pos in lexicon:
        if surface == stem:
            analyses.add(stem + "+" + pos)

        for feature in features[pos]:
            if generate(stem, feature) == surface:
                analyses.add(stem + "+" + pos + "+" + feature)

    if analyses:
        print(", ".join(sorted(analyses)))
    else:
        print("UNKNOWN")


cat+N+PL
watch+N+PL, watch+V+3SG
watch+V+PROG
study+V+PAST
big+ADJ+COMP
cat+N
big+ADJ+SUP
UNKNOWN
